<div align="center">
  <img src="assets/day14_header.png" alt="Databricks 14 Days AI Challenge - Day 14" width="800"/>
</div>

## DAY 14 (22/01/26) – AI-Powered Analytics: Genie & Mosaic AI

### 📚 Learning Objectives
We have built a robust Lakehouse. Now, we unlock its value using **Artificial Intelligence**.
* **Databricks Genie:** A "Text-to-SQL" engine. Instead of writing complex joins, we ask plain English questions. 
* **Mosaic AI:** Integrating Open Source LLMs (like BERT/Llama) directly into our pipeline to analyze unstructured text.
* **Sentiment Analysis:** Turning qualitative customer reviews into quantitative metrics (Positive/Negative scores).

### 🚀 Strategy: "The Voice of the Customer"
1.  **Genie Setup:** We will configure a Genie Space to allow business users to query our **Gold** tables using natural language.
2.  **NLP Pipeline:** We will generate synthetic "Product Reviews" (since our dataset is numerical) and use a pre-trained **Transformer** model to score their sentiment.
3.  **Insight:** We will determine if high-revenue products also have high customer sentiment.

### 🧞‍♂️ Part 1: Databricks Genie (Natural Language to SQL)

*Since Genie is a UI-based feature, follow these steps to "enable" your Lakehouse for AI queries.*

1.  **Navigate:** Click **"Genie"** in the left sidebar.
2.  **Create:** Click **"New Genie Space"**.
3.  **Connect Data:** Select your catalog (`course_catalog`) and schema (`ecommerce_governed`). Add the `gold_product_perf` and `silver_events` tables.
4.  **Ask Questions:** Type the following prompts into the chat bar to test the AI's understanding:

> * "Show me the top 5 brands by total revenue."
> * "What is the conversion rate for the 'electronics' category?"
> * "Plot the daily trend of purchases for Apple products in November."
> * "Find users who viewed items more than 10 times but never purchased."

**Why this matters:** Genie allows you to offload "ad-hoc" requests (e.g., "Can you pull this number for me?") to the AI, freeing you up for deep data science work.

###Part 2 - Mosaic AI (Sentiment Analysis Setup)
**Task**: Install libraries and generate synthetic text data. **Concept**: Our eCommerce dataset is behavioral (clicks/buys). To demonstrate AI, we simulate a "Product Review" dataset for our top-performing products.

In [0]:
# 1. Install Hugging Face Transformers & Torch
%pip install transformers torch

# 2. Restart Python to ensure libraries are loaded
# This clears memory! We must define variables in the NEXT cell.
dbutils.library.restartPython()

In [0]:
import pandas as pd
from pyspark.sql.functions import col

# Re-import libraries (since kernel was restarted)
# ---------------------------------------------------------
# STEP 2: SYNTHETIC DATA CREATION
# ---------------------------------------------------------

# 1. Get Top Products from our Gold Table
top_products = spark.table("course_catalog.ecommerce_governed.gold_product_perf") \
    .orderBy(col("total_revenue").desc()) \
    .limit(5) \
    .select("product_id", "brand", "total_revenue") \
    .toPandas()

# 2. Simulate User Reviews
synthetic_reviews = [
    # Positive Reviews
    {"product_id": top_products.iloc[0]['product_id'], "review_text": "Absolutely love this phone! Fast and battery lasts forever."},
    {"product_id": top_products.iloc[1]['product_id'], "review_text": "Great value for the price. Best investment I made this year."},
    # Negative/Neutral Reviews
    {"product_id": top_products.iloc[2]['product_id'], "review_text": "It's okay, but the screen cracked way too easily. Disappointed."},
    {"product_id": top_products.iloc[3]['product_id'], "review_text": "Terrible delivery experience and the item arrived damaged."},
    {"product_id": top_products.iloc[4]['product_id'], "review_text": "Works as advertised, but a bit expensive for what you get."}
]

df_reviews = pd.DataFrame(synthetic_reviews)

print("✅ Data Regenerated in new Kernel:")
display(df_reviews)


###Running the NLP Model
**Task**: Apply a pre-trained Transformer model to score sentiment. **Concept**: We don't need to train a model from scratch. We use Transfer Learning downloading a model (DistilBERT) that already understands English and applying it to our data.

In [0]:
from transformers import pipeline
import mlflow

# ---------------------------------------------------------
# STEP 3: AI INFERENCE LOOP
# ---------------------------------------------------------

# 1. Load Pre-trained Model
# This creates a local pipeline for sentiment analysis
sentiment_pipeline = pipeline("sentiment-analysis")

# 2. Setup MLflow Experiment
mlflow.set_experiment("/Users/engineeringltctanmay@gmail.com/Day14_NLP_Analysis")

print("🤖 Analyzing Sentiment...")

with mlflow.start_run(run_name="Customer_Voice_Analysis"):
    # Log the architecture we are using
    mlflow.log_param("model_arch", "distilbert-base-uncased-finetuned-sst-2-english")
    
    results = []
    
    # Iterate through our Pandas dataframe
    for index, row in df_reviews.iterrows():
        # The model returns a list like [{'label': 'POSITIVE', 'score': 0.99}]
        # We grab the first (and only) result
        prediction = sentiment_pipeline(row['review_text'])[0]
        
        results.append({
            "product_id": row['product_id'],
            "review_text": row['review_text'],
            "sentiment": prediction['label'],
            "confidence": prediction['score']
        })
        
    # Log a metric: Average Confidence
    avg_conf = sum(r['confidence'] for r in results) / len(results)
    mlflow.log_metric("avg_model_confidence", avg_conf)
    
    # Create Final Result DataFrame
    df_sentiment = pd.DataFrame(results)
    
    print(f"✅ Analysis Complete. Average Model Confidence: {avg_conf:.4f}")

# Display the results
display(df_sentiment)

###Merging AI Insights with Business Data
**Task**: Combine the NLP output with the Gold revenue data. **Concept**: This is the "Holy Grail" of analytics—correlating Qualitative (Feelings) with Quantitative (Revenue).

In [0]:
# 1. Merge Sentiment Data with Product Metadata
# We merge on 'product_id' to see which Brand got the bad review
df_final_insight = pd.merge(df_sentiment, top_products, on="product_id")

# 2. Convert back to Spark for native plotting
spark_insight = spark.createDataFrame(df_final_insight)

print("📊 FINAL INSIGHT: Revenue vs. Customer Sentiment")
display(spark_insight.select("brand", "total_revenue", "sentiment", "review_text", "confidence"))

# VISUALIZATION TIP:
# Create a 'Scatter Plot'.
# X-Axis: total_revenue
# Y-Axis: confidence
# Group/Color By: sentiment (Green for Positive, Red for Negative)

### 🧠 Key Learnings & Takeaways
* **Unstructured Data:** We proved that SQL isn't enough. To understand *why* a product sells (or fails), we need to analyze text using **Transformers**.
* **Genie:** We explored how Natural Language Interfaces (NLI) allow us to "chat" with our Lakehouse, democratizing data access for non-technical users.
* **The Full Stack:** Over 14 days, we went from **Raw CSVs** -> **Delta Lake** -> **Medallion Architecture** -> **MLOps** -> **Generative AI**. You have built a complete, modern Data & AI platform.

### 🎉 CONGRATULATIONS!
You have completed the **14-Days of AI Challenge**.
* **Next Step:** Commit your final code to GitHub.
* **Share:** Post your repository link on LinkedIn with the tag `#DatabricksWithIDC`.